# Re-download Fundamentals v2

The original download (`refinitiv_download.ipynb`) lost **18 of 34 batches** (494 active stocks
including ASII, BBCA, BBRI, KLBF, BRPT, BSDE, CPIN, INDF) because `rd.get_data()` with
batch_size=30 and the full 2008–2025 date range silently timed out.

**Fixes (matching the v2 daily download that worked):**
- **Batch size 10** (was 30)
- **3-year time chunks** (was full 2008–2025 in one shot)
- **2s sleep** between batches
- **3 retries** with exponential backoff
- **Per-chunk CSV checkpoints** — skip chunks already saved
- **Start from 2008** to maximize historical coverage

**API note:** Fundamentals use `rd.get_data()` with `TR.*` fields (DataGrid endpoint).
This is correct for fundamental data — the Historical Pricing endpoint (`rd.get_history()`
without fields) is only for OHLCV prices.

**Run order:** Cell 1 → 2 → 3 → 4 (download) → 5 (retry) → 6 (combine) → 7 (save) → 8 (quality check)

In [8]:
import refinitiv.data as rd
import pandas as pd
import numpy as np
from datetime import datetime
import time
import json
import os
import warnings
warnings.filterwarnings('ignore')

rd.open_session()

<refinitiv.data.session.Definition object at 0x122a50f50 {name='workspace'}>

In [9]:
# ── Config ─────────────────────────────────────────────────────────────
DATA_DIR    = '../data'
BATCH_SIZE  = 10
SLEEP       = 2        # seconds between batches (DataGrid is heavier than Historical Pricing)
MAX_RETRIES = 3
CHUNK_PREFIX = 'fund_chunk'

# 7 chunks: 2008–2025
CHUNKS = [
    (datetime(2008, 1, 1),  datetime(2009, 12, 31)),
    (datetime(2010, 1, 1),  datetime(2011, 12, 31)),
    (datetime(2012, 1, 1),  datetime(2014, 12, 31)),
    (datetime(2015, 1, 1),  datetime(2017, 12, 31)),
    (datetime(2018, 1, 1),  datetime(2020, 12, 31)),
    (datetime(2021, 1, 1),  datetime(2023, 12, 31)),
    (datetime(2024, 1, 1),  datetime(2025, 12, 31)),
]

# Same 12 fields as the original download in refinitiv_download.ipynb
FUND_FIELDS = [
    'TR.CompanyMarketCapitalization.date',
    'TR.CompanyMarketCapitalization',
    'TR.F.ReturnAvgTotAssetsPctTTM.date',
    'TR.F.ReturnAvgTotAssetsPctTTM',
    'TR.IPODate',
    'TR.DPSMean.date',
    'TR.DPSMean',
    'TR.F.BookValuePerShr.date',
    'TR.F.BookValuePerShr',
    'TR.TRBCEconomicSector',
    'TR.SharesOutstanding.date',
    'TR.SharesOutstanding',
]

# Use the full RIC list — we want fundamentals for everything
all_rics = pd.read_csv(f'{DATA_DIR}/idx_ric_list.csv')['RIC'].tolist()

n_batches = (len(all_rics) + BATCH_SIZE - 1) // BATCH_SIZE
print(f'RICs to download: {len(all_rics)}')
print(f'Batches per chunk: {n_batches}')
print(f'Total API calls: {n_batches * len(CHUNKS)}')
print(f'Estimated time: ~{n_batches * len(CHUNKS) * (SLEEP + 1) / 60:.0f} min')

RICs to download: 992
Batches per chunk: 100
Total API calls: 700
Estimated time: ~35 min


In [10]:
def fetch_fund_batch(rics, start, end, retries=MAX_RETRIES):
    """Download monthly fundamentals for a batch of RICs with retries.
    
    Uses rd.get_data() with TR.* fields — this is the DataGrid/UDF endpoint,
    which is the CORRECT endpoint for fundamental data.
    (rd.get_history() without fields is for OHLCV prices only.)
    """
    for attempt in range(1, retries + 1):
        try:
            df = rd.get_data(
                universe=rics,
                fields=FUND_FIELDS,
                parameters={
                    'SDate': start.strftime('%Y-%m-%d'),
                    'EDate': end.strftime('%Y-%m-%d'),
                    'Frq': 'M'
                }
            )
            if df is not None and not df.empty:
                return df, None
            else:
                return None, None
        except Exception as e:
            err = str(e)[:150]
            if attempt < retries:
                wait = SLEEP * (2 ** attempt)
                time.sleep(wait)
            else:
                return None, err
    return None, 'max retries exceeded'


def clean_fund_batch(df):
    """Clean a raw fundamentals DataFrame from rd.get_data().
    
    rd.get_data() with these 12 fields returns 13 columns:
      Instrument, Date, MarketCap, Date.1, ROA, IPODate, Date.2, DPS, Date.3, BVPS, Sector, Date.4, Shares
    
    The multiple Date columns come from the .date suffix fields.
    We rename positionally (same approach as the original notebook that worked for 434 stocks).
    """
    expected_cols = [
        'Instrument', 'Date', 'Market_Cap', 'Date_ROA', 'ROA',
        'IPO_Date', 'Date_DPS', 'DPS_Raw', 'Date_BVPS', 'BVPS',
        'Sector', 'Date_Shares', 'Shares_Outstanding'
    ]
    
    if len(df.columns) != len(expected_cols):
        print(f'    WARNING: expected {len(expected_cols)} cols, got {len(df.columns)}: {list(df.columns)}')
        return None
    
    df = df.copy()
    df.columns = expected_cols
    
    # Drop the extra date columns — we only need the primary Date (from MarketCap.date)
    df = df.drop(columns=['Date_ROA', 'Date_DPS', 'Date_BVPS', 'Date_Shares'])
    
    # Type conversions
    for col in ['Market_Cap', 'ROA', 'DPS_Raw', 'BVPS', 'Shares_Outstanding']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df['IPO_Date'] = pd.to_datetime(df['IPO_Date'], errors='coerce')
    
    # Drop rows with no date (useless)
    df = df.dropna(subset=['Date'])
    
    return df


print('Functions defined.')

Functions defined.


## Download all chunks

Each chunk saves to `fund_chunk_N.csv`. If a chunk file already exists, it is skipped.

Each batch is cleaned immediately (no wide-format concatenation that caused the dedup bug in daily v1).

In [11]:
for chunk_idx, (chunk_start, chunk_end) in enumerate(CHUNKS):
    chunk_file = f'{DATA_DIR}/{CHUNK_PREFIX}_{chunk_idx}.csv'
    
    if os.path.exists(chunk_file):
        existing = pd.read_csv(chunk_file)
        print(f'\nChunk {chunk_idx} ({chunk_start.date()} to {chunk_end.date()}): '
              f'ALREADY DONE — {len(existing):,} rows, '
              f'{existing["Instrument"].nunique()} stocks')
        continue
    
    print(f'\n{"="*65}')
    print(f'  Chunk {chunk_idx}: {chunk_start.date()} to {chunk_end.date()}')
    print(f'{"="*65}')
    
    frames = []
    permanently_failed = []
    total_batches = (len(all_rics) + BATCH_SIZE - 1) // BATCH_SIZE
    
    for i in range(0, len(all_rics), BATCH_SIZE):
        batch_rics = all_rics[i:i + BATCH_SIZE]
        batch_num = i // BATCH_SIZE + 1
        
        df_raw, err = fetch_fund_batch(batch_rics, chunk_start, chunk_end)
        
        if df_raw is not None:
            df_clean = clean_fund_batch(df_raw)
            if df_clean is not None and not df_clean.empty:
                frames.append(df_clean)
        elif err:
            permanently_failed.append({
                'rics': batch_rics, 'error': err
            })
        
        if batch_num % 10 == 0 or batch_num == total_batches:
            print(f'  Batch {batch_num}/{total_batches} — '
                  f'{len(frames)} successful, {len(permanently_failed)} failed')
        
        time.sleep(SLEEP)
    
    if frames:
        chunk_df = pd.concat(frames, ignore_index=True)
        chunk_df = chunk_df.drop_duplicates(subset=['Instrument', 'Date'], keep='first')
        chunk_df = chunk_df.sort_values(['Instrument', 'Date']).reset_index(drop=True)
        chunk_df.to_csv(chunk_file, index=False)
        print(f'\n  Saved {chunk_file}: {len(chunk_df):,} rows, '
              f'{chunk_df["Instrument"].nunique()} stocks')
    else:
        print(f'\n  No data for chunk {chunk_idx}!')
    
    if permanently_failed:
        fail_file = f'{DATA_DIR}/{CHUNK_PREFIX}_{chunk_idx}_failed.json'
        with open(fail_file, 'w') as f:
            json.dump(permanently_failed, f, indent=2)
        print(f'  {len(permanently_failed)} failed batches saved to {fail_file}')

print(f'\n{"="*65}')
print('  ALL CHUNKS COMPLETE')
print(f'{"="*65}')


Chunk 0 (2008-01-01 to 2009-12-31): ALREADY DONE — 6,198 rows, 275 stocks

Chunk 1 (2010-01-01 to 2011-12-31): ALREADY DONE — 9,204 rows, 420 stocks

Chunk 2 (2012-01-01 to 2014-12-31): ALREADY DONE — 16,279 rows, 500 stocks

Chunk 3 (2015-01-01 to 2017-12-31): ALREADY DONE — 18,500 rows, 568 stocks

Chunk 4 (2018-01-01 to 2020-12-31): ALREADY DONE — 22,778 rows, 731 stocks

  Chunk 5: 2021-01-01 to 2023-12-31
  Batch 10/100 — 10 successful, 0 failed
  Batch 20/100 — 20 successful, 0 failed
  Batch 30/100 — 30 successful, 0 failed
  Batch 40/100 — 40 successful, 0 failed
  Batch 50/100 — 50 successful, 0 failed
  Batch 60/100 — 60 successful, 0 failed
  Batch 70/100 — 70 successful, 0 failed
  Batch 80/100 — 80 successful, 0 failed
  Batch 90/100 — 90 successful, 0 failed
  Batch 100/100 — 100 successful, 0 failed

  Saved ../data/fund_chunk_5.csv: 27,186 rows, 887 stocks

  Chunk 6: 2024-01-01 to 2025-12-31
  Batch 10/100 — 10 successful, 0 failed
  Batch 20/100 — 20 successful, 0 fa

## Retry failures

In [12]:
import glob as globmod

fail_files = sorted(globmod.glob(f'{DATA_DIR}/{CHUNK_PREFIX}_*_failed.json'))
if not fail_files:
    print('No failed batches to retry!')
else:
    for ff in fail_files:
        # Extract chunk index from filename like 'fund_chunk_2_failed.json'
        parts = os.path.basename(ff).replace('.json', '').split('_')
        chunk_idx = int(parts[2])
        chunk_start, chunk_end = CHUNKS[chunk_idx]
        
        with open(ff) as f:
            failures = json.load(f)
        
        print(f'\nRetrying {len(failures)} failed batches from chunk {chunk_idx}...')
        recovered = []
        still_failed = []
        
        for fail in failures:
            df_raw, err = fetch_fund_batch(fail['rics'], chunk_start, chunk_end, retries=5)
            if df_raw is not None:
                df_clean = clean_fund_batch(df_raw)
                if df_clean is not None and not df_clean.empty:
                    recovered.append(df_clean)
                    print(f'  Recovered: {fail["rics"][:3]}...')
            elif err:
                still_failed.append(fail)
            time.sleep(SLEEP * 2)
        
        if recovered:
            rec_df = pd.concat(recovered, ignore_index=True)
            rec_df = rec_df.drop_duplicates(subset=['Instrument', 'Date'], keep='first')
            
            chunk_file = f'{DATA_DIR}/{CHUNK_PREFIX}_{chunk_idx}.csv'
            if os.path.exists(chunk_file):
                existing = pd.read_csv(chunk_file, parse_dates=['Date', 'IPO_Date'])
                combined = pd.concat([existing, rec_df], ignore_index=True)
                combined = combined.drop_duplicates(subset=['Instrument', 'Date'], keep='first')
                combined.to_csv(chunk_file, index=False)
            else:
                rec_df.to_csv(chunk_file, index=False)
            print(f'  +{len(rec_df):,} rows recovered')
        
        if still_failed:
            with open(ff, 'w') as f:
                json.dump(still_failed, f, indent=2)
            print(f'  {len(still_failed)} still failed')
        else:
            os.remove(ff)
            print(f'  All recovered!')
    
    print('\nIf batches were recovered, re-run the Combine cell below.')


Retrying 28 failed batches from chunk 0...
  Recovered: ['MDKA.JK', 'MDKI.JK', 'MDLA.JK']...
  Recovered: ['META.JK', 'MFIN.JK^J25', 'MFMI.JK']...
  Recovered: ['MINE.JK', 'MIRA.JK', 'MITI.JK']...
  Recovered: ['MLPT.JK', 'MMIX.JK', 'MMLP.JK']...
  Recovered: ['MPRO.JK', 'MPXL.JK', 'MRAT.JK']...
  Recovered: ['MTFN.JK', 'MTLA.JK', 'MTMH.JK']...
  Recovered: ['MYRX.JK^G25', 'MYRX_p.JK^G25', 'MYTX.JK']...
  Recovered: ['NEST.JK', 'NETV.JK', 'NFCX.JK']...
  Recovered: ['OCAP.JK', 'OILS.JK', 'OKAS.JK']...
  Recovered: ['PAMG.JK', 'PANI.JK', 'PANR.JK']...
  Recovered: ['PDPP.JK', 'PEGE.JK', 'PEHA.JK']...
  Recovered: ['PIPA.JK', 'PJAA.JK', 'PJHB.JK']...
  Recovered: ['PNBN.JK', 'PNBS.JK', 'PNGO.JK']...
  Recovered: ['POLY.JK', 'POOL.JK', 'PORT.JK']...
  Recovered: ['PRAY.JK', 'PRDA.JK', 'PRIM.JK']...
  Recovered: ['PTDU.JK', 'PTIS.JK', 'PTMP.JK']...
  Recovered: ['PTSP.JK', 'PUDP.JK', 'PURA.JK']...
  Recovered: ['RAFI.JK', 'RAJA.JK', 'RALS.JK']...
  Recovered: ['RICY.JK', 'RIGS.JK', 'RIMO.

## Combine all chunks

In [13]:
all_chunks = []
for chunk_idx in range(len(CHUNKS)):
    chunk_file = f'{DATA_DIR}/{CHUNK_PREFIX}_{chunk_idx}.csv'
    if os.path.exists(chunk_file):
        df = pd.read_csv(chunk_file, parse_dates=['Date', 'IPO_Date'])
        all_chunks.append(df)
        print(f'  Loaded {chunk_file}: {len(df):,} rows, {df["Instrument"].nunique()} stocks')
    else:
        print(f'  MISSING: {chunk_file}')

fund = pd.concat(all_chunks, ignore_index=True)
fund = fund.drop_duplicates(subset=['Instrument', 'Date'], keep='first')
fund = fund.sort_values(['Instrument', 'Date']).reset_index(drop=True)

# Forward/back-fill static fields within each instrument
fund['Sector'] = fund['Sector'].replace(r'^\s*$', pd.NA, regex=True)
for col in ['IPO_Date', 'Sector']:
    fund[col] = fund.groupby('Instrument')[col].ffill().bfill()

# Derived variables
fund['Market_Cap_Bil'] = fund['Market_Cap'] / 1e9
fund['Div_Payer'] = fund['DPS_Raw'].gt(0).astype(int)
fund['Months_Since_Listing'] = (
    (fund['Date'] - fund['IPO_Date']).dt.days / 30.44
).round().astype('Int64')

print(f'\nCombined fundamentals: {len(fund):,} rows, {fund["Instrument"].nunique()} stocks')
print(f'Date range: {fund["Date"].min().date()} to {fund["Date"].max().date()}')
print(f'\nCoverage:')
for col in fund.columns:
    pct = fund[col].notna().mean() * 100
    if pct < 100:
        print(f'  {col}: {pct:.1f}%')

  Loaded ../data/fund_chunk_0.csv: 8,491 rows, 374 stocks
  Loaded ../data/fund_chunk_1.csv: 9,204 rows, 420 stocks
  Loaded ../data/fund_chunk_2.csv: 16,279 rows, 500 stocks
  Loaded ../data/fund_chunk_3.csv: 18,500 rows, 568 stocks
  Loaded ../data/fund_chunk_4.csv: 22,778 rows, 731 stocks
  Loaded ../data/fund_chunk_5.csv: 27,186 rows, 887 stocks
  Loaded ../data/fund_chunk_6.csv: 20,655 rows, 952 stocks

Combined fundamentals: 122,951 rows, 952 stocks
Date range: 1999-08-20 to 2025-12-31

Coverage:
  ROA: 92.1%
  DPS_Raw: 18.7%
  BVPS: 99.2%
  Shares_Outstanding: 99.5%


## Save and compare to old version

In [14]:
# Back up old file
old_path = f'{DATA_DIR}/idx_fundamentals.csv'
backup_path = f'{DATA_DIR}/idx_fundamentals_v1.csv'

if os.path.exists(old_path) and not os.path.exists(backup_path):
    old = pd.read_csv(old_path)
    old.to_csv(backup_path, index=False)
    print(f'Old fundamentals backed up to {backup_path}')
    print(f'  Old: {len(old):,} rows, {old["Instrument"].nunique()} stocks')
    print(f'  New: {len(fund):,} rows, {fund["Instrument"].nunique()} stocks')
    print(f'  Gain: +{fund["Instrument"].nunique() - old["Instrument"].nunique()} stocks, '
          f'+{len(fund) - len(old):,} rows')
elif os.path.exists(backup_path):
    old = pd.read_csv(backup_path)
    print(f'  v1 backup already exists: {len(old):,} rows, {old["Instrument"].nunique()} stocks')
    print(f'  New: {len(fund):,} rows, {fund["Instrument"].nunique()} stocks')
    print(f'  Gain: +{fund["Instrument"].nunique() - old["Instrument"].nunique()} stocks')
else:
    print('No old file to compare.')

fund.to_csv(old_path, index=False)
print(f'\nSaved idx_fundamentals.csv')

  v1 backup already exists: 58,475 rows, 434 stocks
  New: 122,951 rows, 952 stocks
  Gain: +518 stocks

Saved idx_fundamentals.csv


## Quality check

In [15]:
print('=== QUALITY CHECK ===')
print()

# Check blue chips that were MISSING in v1
blue_chips = ['ASII.JK', 'BBCA.JK', 'BBRI.JK', 'BMRI.JK', 'TLKM.JK',
              'UNVR.JK', 'HMSP.JK', 'GGRM.JK', 'ICBP.JK', 'KLBF.JK',
              'BRPT.JK', 'BSDE.JK', 'CPIN.JK', 'INDF.JK', 'SMGR.JK']

fund_rics = set(fund['Instrument'].unique())
print('Blue-chip coverage (ASII/BBCA/BBRI/KLBF/BRPT/BSDE/CPIN/INDF were missing in v1):')
for bc in blue_chips:
    if bc in fund_rics:
        n = len(fund[fund.Instrument == bc])
        print(f'  {bc}: {n} monthly obs')
    else:
        print(f'  {bc}: STILL MISSING')

print()

# Check against daily panel
mp_file = f'{DATA_DIR}/idx_master_panel.csv'
if os.path.exists(mp_file):
    mp_rics = set(pd.read_csv(mp_file, usecols=['Instrument'])['Instrument'].unique())
    overlap = fund_rics & mp_rics
    still_missing = mp_rics - fund_rics
    active_missing = [r for r in still_missing if '^' not in r]
    print(f'Panel stocks with fundamentals: {len(overlap)} / {len(mp_rics)} ({len(overlap)/len(mp_rics)*100:.1f}%)')
    print(f'Still missing: {len(still_missing)} total ({len(active_missing)} active, {len(still_missing)-len(active_missing)} delisted)')
else:
    print(f'Master panel not found at {mp_file} — skipping cross-check')

print()

# Stocks per year
fund_copy = fund.copy()
fund_copy['Year'] = fund_copy['Date'].dt.year
print('Stocks with fundamentals by year:')
for y in range(2008, 2026):
    n = fund_copy[fund_copy.Year == y]['Instrument'].nunique()
    if n > 0:
        print(f'  {y}: {n}')

print()

# Sector distribution
print('Sector distribution:')
print(fund.groupby('Sector')['Instrument'].nunique().sort_values(ascending=False).to_string())

print()

# Key field coverage
print('Field coverage (non-null %):')
for col in ['Market_Cap', 'ROA', 'BVPS', 'DPS_Raw', 'Shares_Outstanding', 'Sector', 'IPO_Date']:
    pct = fund[col].notna().mean() * 100
    print(f'  {col}: {pct:.1f}%')

=== QUALITY CHECK ===

Blue-chip coverage (ASII/BBCA/BBRI/KLBF/BRPT/BSDE/CPIN/INDF were missing in v1):
  ASII.JK: 216 monthly obs
  BBCA.JK: 216 monthly obs
  BBRI.JK: 216 monthly obs
  BMRI.JK: 216 monthly obs
  TLKM.JK: 216 monthly obs
  UNVR.JK: 216 monthly obs
  HMSP.JK: 216 monthly obs
  GGRM.JK: 216 monthly obs
  ICBP.JK: 183 monthly obs
  KLBF.JK: 216 monthly obs
  BRPT.JK: 216 monthly obs
  BSDE.JK: 211 monthly obs
  CPIN.JK: 216 monthly obs
  INDF.JK: 216 monthly obs
  SMGR.JK: 216 monthly obs

Master panel not found at ../data/idx_master_panel.csv — skipping cross-check

Stocks with fundamentals by year:
  2008: 358
  2009: 371
  2010: 387
  2011: 416
  2012: 439
  2013: 468
  2014: 494
  2015: 508
  2016: 524
  2017: 559
  2018: 611
  2019: 664
  2020: 713
  2021: 747
  2022: 778
  2023: 837
  2024: 861
  2025: 883

Sector distribution:
Sector
Consumer Cyclicals        170
Industrials               152
Consumer Non-Cyclicals    129
Basic Materials           119
Financials  